In [0]:
import random
import json
from datetime import datetime as dt, timedelta

In [0]:
# Block 1 config 

CYCLES_PER_SHIFT = 35
PAYLOAD_TARGETS = {"777": 90, "785": 140}
PAYLOAD_VARIATION = 0.10
TELEMETRY_PATH = "/Volumes/mining/landing/telemetry"
GROUND_TRUTH_PATH = "/Volumes/mining/ground_truth/truth"


In [0]:
print(PAYLOAD_TARGETS)

In [0]:
# Block 2 The fleet
trucks = [
    {"truck_id": "T01", "model": "777", "capacity": 90},
    {"truck_id": "T02", "model": "777", "capacity": 90},
    {"truck_id": "T03", "model": "777", "capacity": 90},
    {"truck_id": "T04", "model": "777", "capacity": 90},
    {"truck_id": "T05", "model": "785", "capacity": 140},
    {"truck_id": "T06", "model": "785", "capacity": 140},
    {"truck_id": "T07", "model": "785", "capacity": 140},
    {"truck_id": "T08", "model": "785", "capacity": 140},
]

for truck in trucks:
    print(f"Truck ID: {truck['truck_id']}, Model: {truck['model']}, Capacity: {truck['capacity']} tons")


In [0]:
# Block 3 Generate one shift of haul cycles.
# Builds two parallel lists: dirty telemetry (pipeline ingests this) and
# the ground-truth answer key (pipeline must never read this).
# v1: NO corruption yet  reported_tonnage == true_tonnage.

SHIFT_ID = dt.now().strftime("%Y%m%d_%H%M%S")  # unique per run so files don't collide
SHIFT_START = dt.now()
CYCLE_MINUTES = 15  # ~15 min/cycle (2 km haul, load, dump, return)

telemetry_records = []  # -> landing volume
truth_records = []      # -> ground_truth volume

for truck in trucks:
    for cycle_number in range(CYCLES_PER_SHIFT):
        # the one random draw: the TRUE tonnage this load actually was
        target = truck["capacity"]
        low  = target * (1 - PAYLOAD_VARIATION)
        high = target * (1 + PAYLOAD_VARIATION)
        true_tonnage = round(random.uniform(low, high), 1)

        # unique across trucks AND shifts -> safe dedup key later
        cycle_id = f"{SHIFT_ID}-{truck['truck_id']}-C{cycle_number:02d}"

        cycle_time = SHIFT_START + timedelta(minutes=cycle_number * CYCLE_MINUTES)
        timestamp = cycle_time.isoformat()

        telemetry_records.append({
            "cycle_id": cycle_id,
            "shift_id": SHIFT_ID,        # <-- new: shift as a first-class field
            "truck_id": truck["truck_id"],
            "model": truck["model"],
            "timestamp": timestamp,
            "reported_tonnage": true_tonnage,  # v1: equals truth
        })

        truth_records.append({
            "cycle_id": cycle_id,
            "shift_id": SHIFT_ID,        # <-- new: needed for per-shift reconciliation
            "truck_id": truck["truck_id"],
            "true_tonnage": true_tonnage,
        })

print(f"Generated {len(telemetry_records)} telemetry and {len(truth_records)} truth records for shift {SHIFT_ID}")

In [0]:
# Block 4 Write telemetry as NDJSON (one JSON object per line).
telemetry_file = f"{TELEMETRY_PATH}/telemetry_{SHIFT_ID}.json"

with open(telemetry_file, "w") as f:
    for record in telemetry_records:
        f.write(json.dumps(record) + "\n")

print(f"Wrote {len(telemetry_records)} records to {telemetry_file}")

In [0]:
# Block 5 Write answer key to the walled-off ground_truth volume.
truth_file = f"{GROUND_TRUTH_PATH}/truth_{SHIFT_ID}.json"

with open(truth_file, "w") as f:
    for record in truth_records:
        f.write(json.dumps(record) + "\n")

print(f"Wrote {len(truth_records)} records to {truth_file}")

In [0]:
# Block 6 Peek at the first 3 telemetry lines to confirm shape.
with open(telemetry_file) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line.strip())